# 🧩 Mini-Lab: Define the Interface, Keep the Mock

**Module 6: The Replaceable Model Interface** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Locate** where a vendor call leaked into business logic and mark the seam
2. **Write** an `AIClient` Protocol so backends are interchangeable
3. **Make** the existing mock conform and inject it into the pipeline

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Interface seam | Small typed contract; adapter returns your dataclass, never their object |
| Dependency injection | Pass the backend in; callers choose, pipeline code stays unchanged |


## The Scenario

Imagine you inherited a small codebase that classifies customer support tickets. The original developer hard-coded the OpenAI API call **directly inside** the business logic. One day the vendor changes its API, raises prices, or goes down — and the entire application breaks.

We'll look at this "bad" code, pinpoint exactly where the vendor dependency leaks in, and then mark where the **seam** should go.

## Step 1 — The Tightly-Coupled Code

Below is a trimmed-down ticket classifier. Notice how many lines **know about the vendor** (OpenAI-specific imports, model names, response shapes).

In [1]:
# ❌ Tightly-coupled code — the vendor is everywhere (trimmed to the key lines)

COUPLED_CODE = '''
import openai                                       # 🔴 vendor import

def classify_ticket(ticket_text: str) -> str:
    prompt = f"Classify this support ticket: {ticket_text}"   # business logic: build the prompt

    client = openai.OpenAI()                        # 🔴 vendor client
    response = client.chat.completions.create(      # 🔴 vendor API shape
        model="gpt-4o-mini",                        # 🔴 vendor model name
        messages=[{"role": "user", "content": prompt}],
    )
    raw = response.choices[0].message.content       # 🔴 vendor response shape

    return raw.strip().lower()                      # business logic: clean the answer
'''

print(COUPLED_CODE)


import openai                                       # 🔴 vendor import

def classify_ticket(ticket_text: str) -> str:
    prompt = f"Classify this support ticket: {ticket_text}"   # business logic: build the prompt

    client = openai.OpenAI()                        # 🔴 vendor client
    response = client.chat.completions.create(      # 🔴 vendor API shape
        model="gpt-4o-mini",                        # 🔴 vendor model name
        messages=[{"role": "user", "content": prompt}],
    )
    raw = response.choices[0].message.content       # 🔴 vendor response shape

    return raw.strip().lower()                      # business logic: clean the answer



In [2]:
# Which lines would break if OpenAI disappeared tomorrow? A few examples:
vendor_lines = [
    "import openai",
    "client.chat.completions.create(...)",
    "response.choices[0].message.content",
]

print("Vendor touch-points (examples):")
for line in vendor_lines:
    print(f"  🔴  {line}")

print("\nEach one ties the function to one vendor's package, API method, or response shape.")

Vendor touch-points (examples):
  🔴  import openai
  🔴  client.chat.completions.create(...)
  🔴  response.choices[0].message.content

Each one ties the function to one vendor's package, API method, or response shape.


These vendor-specific lines are **mixed right in** with the business logic (building the prompt, cleaning the answer). That's the problem: there is no boundary, no **seam**, between *what the app does* and *which AI service it calls*.

## Step 2 — What Breaks When the Vendor Changes?

Let's make this concrete. Suppose the vendor changes its response format. This really happens when APIs release new versions.

In [3]:
# Simulating a vendor API change

# OLD response shape (v1) — what the coupled code was written for
old_response = {"choices": [{"message": {"content": "billing"}}]}

# NEW response shape (v2) — the vendor renamed the fields
new_response = {"output": [{"text": "billing"}]}

# The coupled code still digs into the OLD shape, so the new one breaks it
try:
    result = new_response["choices"][0]["message"]["content"]
    print(f"✅ New format works: '{result}'")
except (KeyError, IndexError) as e:
    print(f"❌ New format broke with {e!r}. The business logic didn't change, but classify_ticket() fails anyway.")

❌ New format broke with KeyError('choices'). The business logic didn't change, but classify_ticket() fails anyway.


## Step 3 — Drawing the Seam

An **interface seam** is the line you draw between "business logic" and "vendor call." Everything above the seam is *your application*. Everything below the seam is *the replaceable AI backend*.

Let's label each part of the original function.

> 🧩 **Vendor seam / Protocol interface.** The typed contract from the video; here we mark where it goes.

In [4]:
from IPython.display import display, Markdown


def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))


# Labeling each section of classify_ticket()
diagram = """
## Function Anatomy with Seam

```
def classify_ticket(ticket_text):
    │  BUSINESS: define categories, build the prompt
    │  ═══════════ ✂ SEAM ✂ ═══════════
    │  VENDOR:   OpenAI client, chat.completions.create(),
    │            response.choices[0]...   (replaceable)
    │  ═══════════ ✂ SEAM ✂ ═══════════
    │  BUSINESS: validate the answer, return the category
```

The vendor block in the middle is the part you extract behind an interface.
"""

md(diagram)


## Function Anatomy with Seam

```
def classify_ticket(ticket_text):
    │  BUSINESS: define categories, build the prompt
    │  ═══════════ ✂ SEAM ✂ ═══════════
    │  VENDOR:   OpenAI client, chat.completions.create(),
    │            response.choices[0]...   (replaceable)
    │  ═══════════ ✂ SEAM ✂ ═══════════
    │  BUSINESS: validate the answer, return the category
```

The vendor block in the middle is the part you extract behind an interface.


## Step 4 — What the Seam Enables

Once you extract the vendor block behind an interface, the business logic only knows:

> *"I give you a prompt string, you give me back a response string."*

It doesn't care **who** answers. It could be OpenAI, Anthropic, a local Ollama model, or a mock that returns `"billing"` every time for testing.

In [5]:
# After placing the seam: only the signature and the vendor call change
md("""
```python
def classify_ticket(ticket_text: str, ai_client) -> str:
    ...
    raw = ai_client.complete(prompt)   # ← the interface seam
```
**Zero vendor imports.** Any backend with `.complete(prompt)` works.
""")


```python
def classify_ticket(ticket_text: str, ai_client) -> str:
    ...
    raw = ai_client.complete(prompt)   # ← the interface seam
```
**Zero vendor imports.** Any backend with `.complete(prompt)` works.


In [6]:
checklist = [
    ("import openai / import anthropic in business-logic files",
     "Vendor import should only appear in the backend module"),
    ("Model name hard-coded in the function",
     "Model selection belongs in configuration, not logic"),
    ("response.choices[0].message.content deep in your code",
     "Response parsing belongs inside the backend adapter"),
    ("Cannot run tests without a real API key",
     "A seam lets you inject a mock — no key needed"),
    ("Changing provider requires editing many files",
     "With a seam, you only change the backend module"),
]

print("🔍 Missing-Seam Checklist\n")
for i, (symptom, fix) in enumerate(checklist, 1):
    print(f"  {i}. Symptom:  {symptom}")
    print(f"     Fix:      {fix}\n")

🔍 Missing-Seam Checklist

  1. Symptom:  import openai / import anthropic in business-logic files
     Fix:      Vendor import should only appear in the backend module

  2. Symptom:  Model name hard-coded in the function
     Fix:      Model selection belongs in configuration, not logic

  3. Symptom:  response.choices[0].message.content deep in your code
     Fix:      Response parsing belongs inside the backend adapter

  4. Symptom:  Cannot run tests without a real API key
     Fix:      A seam lets you inject a mock — no key needed

  5. Symptom:  Changing provider requires editing many files
     Fix:      With a seam, you only change the backend module



## Recap

In the **seam teardown** above (Steps 1–4), we identified the problem: vendor-specific code (imports, model names, response parsing) was tangled with business logic. We drew a ✂ seam line showing *where* to cut.

Now we'll **build what goes at that seam** — a small typed interface (a Python `Protocol`) and the dependency-injection pattern to wire it up.

## Step 1 — Define the Data Shapes

Before defining the interface, we need to agree on **what goes in** and **what comes out**. Dataclasses give us lightweight, typed containers for this.

In [7]:
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class AIRequest:
    """What the business logic sends to the AI backend."""
    prompt: str
    system: str = "You are a helpful assistant."
    temperature: float = 0.0
    max_tokens: int = 256


@dataclass
class AIResponse:
    """What the AI backend returns to the business logic."""
    text: str
    model: str = "unknown"
    usage: dict = field(default_factory=dict)

**Why dataclasses?**  
- They are self-documenting: field names and types are visible at a glance.  
- They enforce a consistent shape — every backend receives the same `AIRequest` and returns the same `AIResponse`.  
- Default values (like `temperature=0.0`) keep call sites clean.

## Step 2 — Define the Protocol (The Interface)

A `Protocol` is Python's way of saying *"any class with this method signature is compatible."* No inheritance required — if it has the right method, it fits.

In [8]:
from typing import Protocol, runtime_checkable


@runtime_checkable
class AIClient(Protocol):
    """Any AI backend must implement this single method."""

    def complete(self, request: AIRequest) -> AIResponse:
        """Send a request and get a response."""
        ...

**Key points:**
- `@runtime_checkable` lets us use `isinstance()` to verify a backend at runtime.
- The protocol has exactly **one method**: `complete`. This is the seam we marked in the teardown above.
- Any class that implements `complete(AIRequest) -> AIResponse` is a valid `AIClient` — no base class needed.

## Step 3 — Build a Mock Backend

We'll create a **mock** backend for testing. It satisfies the `AIClient` protocol without inheriting from it.

In [9]:
class MockAIClient:
    """Always returns a fixed answer — perfect for testing."""

    def __init__(self, fixed_answer: str = "billing"):
        self.fixed_answer = fixed_answer

    def complete(self, request: AIRequest) -> AIResponse:
        return AIResponse(text=self.fixed_answer, model="mock-1.0")


# Verify the mock satisfies the protocol at runtime
mock = MockAIClient()
print(f"MockAIClient satisfies AIClient? {isinstance(mock, AIClient)}")

MockAIClient satisfies AIClient? True


`MockAIClient` never inherits from `AIClient`. A `Protocol` uses **structural typing**: if the method signature matches, the contract is satisfied.

## Step 4 — Dependency Injection in Action

Now let's rewrite the ticket classifier from the teardown above — but this time, the AI backend is **injected** as a parameter. The function has zero vendor imports.

> 🧩 **Injecting the client.** The pattern from the video, now wired into our classifier.

In [10]:
def classify_ticket(ticket_text: str, ai_client: AIClient) -> str:
    """Classify a support ticket. The AI backend is injected."""
    categories = ["billing", "technical", "account", "other"]
    request = AIRequest(
        prompt=(
            f"Classify this support ticket into one of {categories}.\n"
            f"Ticket: {ticket_text}\nCategory:"
        ),
    )

    # The seam: one call, no vendor details
    response = ai_client.complete(request)

    category = response.text.strip().lower()
    if category not in categories:
        category = "other"
    return category


# Use the mock backend — no API key, no network
result = classify_ticket("I was charged twice this month", mock)
print(f"Mock backend → {result}")

Mock backend → billing


**What just happened:**
- `classify_ticket` has **zero vendor imports**.
- The AI backend is a parameter (`ai_client`). That's dependency injection.
- Swapping the mock for a real hosted backend later means changing **one argument**, not editing the function.

That's the whole pattern: one Protocol, two dataclass shapes, and a backend injected as a parameter.

## 🎯 Summary

### Key Takeaways

1. **Protocol as interface** — a Python `Protocol` defines the contract (`complete(AIRequest) -> AIResponse`) that any AI backend must satisfy, without requiring inheritance
2. **Dataclass shapes** — `AIRequest` and `AIResponse` dataclasses make the data flowing across the seam explicit, typed, and self-documenting
3. **Dependency injection** — passing the `ai_client` as a parameter means the business logic never imports or knows about any specific vendor
4. **Instant testability** — a `MockAIClient` that returns a fixed answer lets you test the entire pipeline without an API key or network access